# Seance 8 - Faire parler les donnees

**Formation Python 360 - Commission Scientifique nationale - ASEGUIM**

Derniere seance du cursus.

---

## Avant de commencer

Dans Colab : **Fichier > Enregistrer une copie dans Drive**. Sans cela, tu
travailles dans un fichier en lecture seule et rien ne sera garde.

Ce notebook reprend, dans l'ordre, les six fichiers du dossier `reprise/`.
Il se lit de haut en bas : chaque cellule suppose que les precedentes ont
ete executees.

## Ce qu'on fait aujourd'hui

1. Agreger : `groupby`, `reset_index`, `agg`, `pivot_table`
2. Joindre : `merge`, les quatre jointures, et le piege de l'explosion
3. Dessiner : Figure et Axes, le squelette en six lignes, l'export
4. seaborn : le catalogue des traces, `hue`, les facettes
5. Ne pas mentir : les cinq regles de la dataviz honnete

## Les trois pieges de la journee

1. `merge` peut faire grossir ta table sans rien dire.
2. `how="inner"` fait disparaitre des lignes en silence.
3. `sns.barplot` trace une moyenne, pas une somme.

## La regle du jour

Le titre porte le message, pas la description. Pour chaque graphique :
qu'est-ce que ce graphique m'apprend, en une phrase ? Cette phrase est le
titre.

---

## 0. Les outils et les donnees

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

print("pandas     :", pd.__version__)
print("numpy      :", np.__version__)
print("matplotlib :", matplotlib.__version__)
print("seaborn    :", sns.__version__)

# Il faut au moins matplotlib 3.10 et seaborn 0.13.
# Si une version est trop ancienne dans Colab :
#     !pip install -q "matplotlib>=3.10" "seaborn>=0.13"
# puis Execution > Redemarrer la session.

Deux tables aujourd'hui, et c'est tout le sujet de la seance.

| Fichier | Ce que c'est |
|---|---|
| `releve_propre.csv` | les 333 depenses nettoyees en seance 7 |
| `budget_prevu.csv` | ce qu'on avait **prevu** de depenser |

La fonction ci-dessous cherche le fichier en local (si tu travailles dans
le depot) et le telecharge depuis GitHub sinon (si tu es dans Colab).

In [ ]:
from pathlib import Path

BASE = "https://raw.githubusercontent.com/8sylla/python-360/main/data/"


def source(nom):
    """Rend le chemin local du fichier s'il existe, sinon son URL."""
    for dossier in [Path.cwd(), *Path.cwd().parents]:
        candidat = dossier / "data" / nom
        if candidat.exists():
            return candidat
    return BASE + nom


df = pd.read_csv(source("releve_propre.csv"), parse_dates=["date_operation"])
budget = pd.read_csv(source("budget_prevu.csv"), sep=";")

print("releve :", df.shape)
print("budget :", budget.shape)

Le releve est deja propre : c'est le travail de la seance 7. On part donc
directement de donnees exploitables.

`parse_dates` n'est pas optionnel : sans lui, la colonne des dates revient
en texte et `.dt` ne marche plus.

In [ ]:
df.head()

In [ ]:
budget.head()

Note la forme du budget : **une ligne par couple (categorie, mois)**. Six
categories sur six mois, donc 36 lignes. Ce detail va compter.

---

# 1. Echauffement

Sept questions. Pour chacune, ecris ta reponse sur un papier **avant**
d'executer. Le but n'est pas d'avoir raison, c'est de savoir ou tu hesites.

**[1]** Que rend `df.groupby("categorie")["montant"].sum()` : un DataFrame
ou une Series ?

In [ ]:
resultat = df.groupby("categorie")["montant"].sum()
print("type  :", type(resultat).__name__)
print("index :", list(resultat.index))

Une **Series**, indexee par la categorie. C'est pour cela que seaborn exige
un `.reset_index()` derriere : il dessine des colonnes, pas des index.

**[2]** Combien de lignes apres ce `merge` ?

In [ ]:
gauche = pd.DataFrame({"cle": ["x", "y"], "valeur": [1, 2]})
droite = pd.DataFrame({"cle": ["x", "x", "x", "y"], "info": ["a", "b", "c", "d"]})

fusion = gauche.merge(droite, on="cle")
print("gauche :", len(gauche), "lignes")
print("droite :", len(droite), "lignes")
print("fusion :", len(fusion), "lignes")

La cle `x` apparait trois fois a droite : la ligne `x` de gauche est
dupliquee trois fois. Aucun avertissement.

Le reflexe a installer des maintenant : **compter les lignes avant et
apres**. Toujours.

**[3]** `inner` ou `left` : quelle difference, concretement ?

In [ ]:
budget_partiel = pd.DataFrame({"cle": ["x"], "prevu": [100]})

print("inner :", len(gauche.merge(budget_partiel, on="cle", how="inner")), "ligne(s)")
print("left  :", len(gauche.merge(budget_partiel, on="cle", how="left")), "ligne(s)")
print()
print(gauche.merge(budget_partiel, on="cle", how="left"))

`inner` a fait **disparaitre** la ligne `y`, sans le dire. Sur un releve,
c'est une depense perdue. `how="left"` la garde, avec un `NaN` visible a la
place du budget manquant.

**[4]** `plt.bar()` ou `ax.bar()` : pourquoi cela compte ?

In [ ]:
fig, (a1, a2) = plt.subplots(1, 2, figsize=(8, 3))
a1.bar(["A", "B"], [3, 5])
a2.bar(["A", "B"], [5, 3])
print("la figure contient", len(fig.axes), "Axes")
plt.show()

Avec `plt.bar()`, sur **lequel des deux** aurait-on dessine ? Sur le
graphique courant, c'est-a-dire le dernier cree. Invisible dans le code,
donc impraticable des qu'il y en a deux.

**[5]** Que valent la moyenne et la mediane de cette petite serie ?

In [ ]:
loyers = pd.Series([40.0, 55.0, 60.0, 45.0, 900.0])
print("moyenne :", loyers.mean())
print("mediane :", loyers.median())

Une seule grosse valeur deplace la moyenne de 50 a 220. La mediane, elle,
ne bouge pas. Quand les deux s'ecartent, c'est le signal qu'il y a des
valeurs extremes.

**[6]** Un axe qui ne part pas de zero : de combien mens-tu ?

In [ ]:
valeurs = [98.0, 100.0, 102.0]
ecart_reel = (max(valeurs) - min(valeurs)) / min(valeurs) * 100
print(f"ecart reel entre la plus petite et la plus grande : {ecart_reel:.0f} %")

Mais si l'axe part de 97, la barre de droite parait deux fois et demie plus
haute que celle de gauche. Un ecart visuel de 150 % pour un ecart reel de
4 %.

Ce n'est pas une erreur technique : c'est un choix, et il t'engage.

**[7]** Trier par ordre alphabetique, ou par valeur ?

In [ ]:
scores = pd.DataFrame({"pays": ["Algerie", "Benin", "Cameroun"],
                       "note": [12.0, 45.0, 8.0]})
print("alphabetique :", list(scores["pays"]))
print("par valeur   :", list(scores.sort_values("note", ascending=False)["pays"]))

L'ordre alphabetique n'apporte aucune information : il oblige le lecteur a
comparer les longueurs de barres lui-meme.

Exception : ce qui a deja un ordre naturel - les mois, les tranches d'age.

Combien de fois as-tu hesite ? C'est exactement le programme de la seance.

---

# 2. Agreger

Trois gestes, toujours dans le meme ordre : **separer** en piles,
**calculer** sur chaque pile, **recoller** les resultats.

C'est le modele decouper - appliquer - combiner.

In [ ]:
paquets = df.groupby("categorie")

print("objet rendu :", type(paquets).__name__)
print("nombre de paquets :", paquets.ngroups)
print()
print(paquets.size())

`groupby` seul ne calcule **rien**. Il prepare des paquets et attend. C'est
l'agregation qui declenche le travail.

In [ ]:
df.groupby("categorie")["montant"].sum().sort_values(ascending=False)

Six chiffres remplacent 333 lignes.

## 2.1 reset_index : de l'index a la colonne

In [ ]:
totaux = df.groupby("categorie")["montant"].sum()

print("SANS reset_index")
print("  colonnes :", list(pd.DataFrame(totaux).columns))
print("  index    :", totaux.index.name)

table = totaux.sort_values(ascending=False).reset_index(name="total")
print()
print("AVEC reset_index")
print("  colonnes :", list(table.columns))
table

seaborn dessine des **colonnes**, pas des index. Sans `reset_index`, il ne
trouve pas `categorie` et leve une erreur peu claire.

La regle : toute agregation destinee a un graphique finit par
`reset_index()`.

## 2.2 Plusieurs calculs d'un coup

La forme `nom=("colonne", "fonction")` nomme les colonnes de sortie. C'est
la seule a retenir : l'autre produit des index a plusieurs niveaux,
illisibles.

In [ ]:
resume = (
    df.groupby("categorie")
    .agg(
        nombre=("montant", "count"),
        total=("montant", "sum"),
        moyenne=("montant", "mean"),
        mediane=("montant", "median"),
        maximum=("montant", "max"),
    )
    .sort_values("total", ascending=False)
    .reset_index()
)
resume.round(2)

Lis la ligne `Transport` et la ligne `Logement` a voix haute.

Le transport est le poste le **plus frequent** et le **moins cher**. Le
logement est le moins frequent et de loin le plus lourd.

Une moyenne seule ment. Un `count` pose a cote d'elle dit la verite.

Regarde aussi la ligne `Autre` : sa moyenne est nettement au-dessus de sa
mediane. C'est le signe qu'une valeur atypique la tire vers le haut.

## 2.3 Deux formes pour le meme calcul

In [ ]:
longue = df.groupby(["categorie", "mois"])["montant"].sum().reset_index(name="total")
print("FORME LONGUE :", longue.shape)
longue.head(6).round(2)

In [ ]:
croise = df.pivot_table(index="categorie", columns="mois",
                        values="montant", aggfunc="sum", fill_value=0)
print("FORME CROISEE :", croise.shape)
croise.round(0)

Memes chiffres. Mais personne ne repere dans la liste de 36 lignes que la
categorie `Autre` explose en avril - alors que cela saute aux yeux dans le
tableau croise.

Choisir la forme, c'est choisir ce que le lecteur verra.

---

# 3. Joindre

« 4 600 euros de loisirs », est-ce beaucoup ? Le releve ne peut pas
repondre : il ne contient pas ce qu'on avait prevu de depenser.

On ne joint pas deux tables pour apprendre `merge`. On les joint parce
qu'aucune des deux ne repond seule a la question posee.

## 3.1 merge, c'est le RECHERCHEV

In [ ]:
reel = df.groupby(["categorie", "mois"])["montant"].sum().reset_index(name="reel")
fusion = reel.merge(budget, on=["categorie", "mois"], how="left")

print("reel   :", len(reel), "lignes")
print("budget :", len(budget), "lignes")
print("fusion :", len(fusion), "lignes   <- identique a reel : bon signe")
fusion.head(6).round(2)

« En mieux » que RECHERCHEV pour trois raisons : la cle peut etre composee
de **plusieurs colonnes**, le type de jointure est **explicite**, et rien ne
casse quand on insere une colonne au milieu du tableau.

## 3.2 Le piege : l'explosion de lignes

On oublie `mois` dans la cle. C'est une erreur tres facile a faire.

In [ ]:
explose = df.merge(budget, on="categorie", how="left")

print("avant :", len(df), "lignes")
print("apres :", len(explose), "lignes")
print(f"facteur : x{len(explose) / len(df):.0f}")
print()
print(f"total juste  : {df['montant'].sum():12,.2f} EUR")
print(f"total fausse : {explose['montant'].sum():12,.2f} EUR")

Chaque depense a ete dupliquee **six** fois, une par mois du budget. Aucune
erreur, aucun avertissement : juste un total six fois trop grand, et
parfaitement plausible si on ne le verifie pas.

Le plus sournois : les **moyennes ne changent pas** (la moyenne d'un tableau
duplique est identique), et les **proportions non plus**. Une bonne partie
des verifications qu'on fait spontanement passent au vert.

La seule qui echoue est le comptage de lignes.

## 3.3 Le reflexe, en trois lignes

In [ ]:
avant = len(reel)
verifie = reel.merge(budget, on=["categorie", "mois"], how="left")
assert len(verifie) == avant, f"explosion : {avant} -> {len(verifie)}"
print("jointure verifiee :", len(verifie), "lignes")

Et la version que pandas offre lui-meme :

In [ ]:
try:
    df.merge(budget, on="categorie", how="left", validate="many_to_one")
except pd.errors.MergeError as erreur:
    print("MergeError :", str(erreur).split("\n")[0])

`validate="many_to_one"` dit : la cle doit etre unique a droite. Si elle ne
l'est pas, cela **plante** au lieu de gonfler en silence.

La difference est importante : le comptage **detecte** apres coup,
`validate` **empeche** avant.

## 3.4 Les quatre jointures

In [ ]:
depenses_ex = pd.DataFrame({"categorie": ["Logement", "Loisirs", "Voyage"],
                            "reel": [6000, 900, 400]})
prevus_ex = pd.DataFrame({"categorie": ["Logement", "Loisirs", "Sante"],
                          "prevu": [6200, 600, 500]})

for comment in ["inner", "left", "right", "outer"]:
    resultat = depenses_ex.merge(prevus_ex, on="categorie", how=comment)
    print(f"how='{comment:5}' -> {len(resultat)} lignes : "
          f"{', '.join(resultat['categorie'])}")

- `inner` : seulement ce qui existe des **deux** cotes. Voyage et Sante
  disparaissent, sans un mot. **C'est le defaut de merge.**
- `left` : tout le reel, complete si possible. Voyage reste, avec un `NaN`
  visible. C'est presque toujours ce qu'on veut.
- `right` : l'inverse. Rarement utile.
- `outer` : tout le monde. Utile pour **auditer**.

In [ ]:
audit = depenses_ex.merge(prevus_ex, on="categorie", how="outer", indicator=True)
audit

`left_only` veut dire « une depense sans budget prevu ». `right_only`, « un
budget jamais depense ». Les deux sont des informations, pas des erreurs.

## 3.5 Ce que la jointure nous apprend

In [ ]:
bilan = (
    fusion.groupby("categorie")[["reel", "budget_prevu"]].sum().reset_index()
)
bilan["ecart"] = bilan["reel"] - bilan["budget_prevu"]
bilan["ecart_pct"] = (bilan["reel"] / bilan["budget_prevu"] - 1) * 100
bilan.sort_values("ecart_pct", ascending=False).round(2)

Regarde les trois ecarts d'environ 700 euros. En euros, ces lignes se
ressemblent. En pourcentage, deux sont des depassements nets a corriger et
la troisieme est un budget tenu.

700 euros est un probleme sur un budget de 4 000, et du bruit sur un budget
de 37 000.

---

# 4. Matplotlib : le cadre et la photo

La **Figure** est le cadre, la feuille entiere. L'**Axes** est la photo a
l'interieur, la zone de trace. Un cadre peut contenir plusieurs photos.

On enregistre la Figure, jamais un Axes : c'est le cadre qu'on accroche.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
print("fig :", type(fig).__name__, "- le cadre")
print("ax  :", type(ax).__name__, "- la photo")
plt.close(fig)

grille, axes = plt.subplots(2, 2, figsize=(8, 6))
print()
print("plt.subplots(2, 2) ->", len(grille.axes), "Axes, de forme", axes.shape)
plt.close(grille)

## 4.1 Le squelette, en six lignes

C'est tout ce qu'il y a a memoriser.

In [ ]:
table = (
    df.groupby("categorie")["montant"].sum()
    .sort_values(ascending=False).reset_index(name="total")
)

fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(table["categorie"], table["total"], color="#8B7B78")
ax.invert_yaxis()
ax.set_xlabel("Total depense sur six mois (EUR)")
ax.set_title("Depenses par categorie", loc="left", weight="bold")
fig.tight_layout()
plt.show()

Un graphique sans titre d'axe ni unite n'est pas un graphique : c'est une
decoration. Les deux lignes `set_xlabel` et `set_title` ne sont pas
optionnelles.

Pourquoi `barh` et pas `bar` ? Parce que les libelles sont des **mots** : a
l'horizontale ils se lisent sans tourner la tete. Et `invert_yaxis()` met la
plus grosse barre en haut, comme on lit un classement.

Le piege de nommage, en passant d'une documentation a l'autre :
`plt.title()` devient `ax.set_title()`, `plt.xlabel()` devient
`ax.set_xlabel()`. Le prefixe `set_` apparait.

## 4.2 L'axe qui ment

Le meme jeu de donnees, deux fois. Seul l'axe change.

In [ ]:
par_mois = df.groupby("mois")["montant"].sum()

fig, (gauche, droite) = plt.subplots(1, 2, figsize=(13, 4.5))

gauche.bar(par_mois.index, par_mois.values, color="#C1121F")
gauche.set_ylim(par_mois.min() * 0.98, par_mois.max() * 1.02)
gauche.set_title("Axe tronque", loc="left", weight="bold")

droite.bar(par_mois.index, par_mois.values, color="#8B7B78")
droite.set_ylim(0, par_mois.max() * 1.05)
droite.set_title("Axe honnete", loc="left", weight="bold")

for un_axe in (gauche, droite):
    un_axe.set_ylabel("Total du mois (EUR)")
    un_axe.tick_params(axis="x", labelrotation=45)

fig.tight_layout()
plt.show()

ecart = (par_mois.max() / par_mois.min() - 1) * 100
print(f"ecart reel entre le mois le plus faible et le plus fort : {ecart:.0f} %")

A gauche, le mois le plus faible ressemble a un effondrement. C'est le meme
chiffre qu'a droite.

**Regle : l'axe des barres part de zero. Toujours.**

La nuance, parce que quelqu'un la posera : une **courbe** peut se permettre
un axe tronque, elle montre une variation. Une **barre**, non : sa longueur
*est* la quantite. C'est pourquoi les graphiques boursiers ne partent jamais
de zero, et ce sont des courbes.

## 4.3 Exporter

Trois arguments, trois raisons.

In [ ]:
FIGURES = Path("figures")
FIGURES.mkdir(exist_ok=True)

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(par_mois.index, par_mois.values, marker="o", color="#C1121F")
ax.set_ylim(0, par_mois.max() * 1.1)
ax.set_ylabel("Total du mois (EUR)")
ax.set_title("Juin est le mois le plus lourd", loc="left", weight="bold")
fig.tight_layout()

fig.savefig(FIGURES / "sans_soin.png")
fig.savefig(FIGURES / "avec_soin.png", dpi=150, bbox_inches="tight")
fig.savefig(FIGURES / "vectoriel.svg", bbox_inches="tight")
plt.close(fig)

for fichier in sorted(FIGURES.glob("*")):
    print(f"  {fichier.name:18} {fichier.stat().st_size // 1024:4} ko")

- `dpi=150` : net a l'impression et au videoprojecteur. Par defaut c'est
  100, et c'est flou des qu'on agrandit.
- `bbox_inches="tight"` : supprime la marge blanche parasite autour.
- `.svg` : vectoriel, ne pixelise jamais. A preferer pour un rapport ou une
  slide. Le PNG reste utile pour un mail ou une page web.

Le piege du SVG : sur un nuage de 300 000 points, il contient 300 000
formes et devient enorme. Au-dela de quelques milliers d'elements, le PNG a
`dpi` eleve est le bon choix.

Dans Colab, les fichiers ecrits disparaissent a la fin de la session. Pour
les garder : le panneau **Fichiers** a gauche, puis telecharger.

---

# 5. seaborn

seaborn ne remplace pas Matplotlib : il **ecrit du Matplotlib** pour toi, en
y ajoutant les statistiques. Et il dessine dans le `ax` qu'on lui donne, donc
on garde tous les reglages qu'on connait deja.

In [ ]:
sns.set_theme(style="whitegrid")

fig, ax = plt.subplots(figsize=(9, 5))
sns.barplot(data=df, x="montant", y="categorie", ax=ax,
            color="#8B7B78", errorbar=("ci", 95))
ax.set_title("Depense MOYENNE par categorie, et son incertitude",
             loc="left", weight="bold")
ax.set_xlabel("Montant moyen d'une depense (EUR)")
ax.set_ylabel("")
fig.tight_layout()
plt.show()

**Attention, piege.** Par defaut `sns.barplot` trace la **moyenne**, pas la
somme. Les petits traits au bout des barres sont l'intervalle de confiance
a 95 %.

Compare : le total du logement est d'environ 36 000 euros, sa moyenne par
operation d'environ 594. Si tu croyais voir des totaux, tu lis des chiffres
soixante fois plus petits - et coherents entre eux, donc rien ne detonne.

Pour des totaux : agreger d'abord, puis `ax.barh()`.

In [ ]:
# Ce que l'intervalle de confiance raconte
df.groupby("categorie")["montant"].agg(["count", "mean", "std"]).round(2)

Regarde la ligne `Autre` : son ecart-type est **plus grand que sa moyenne**.
Sa barre d'erreur est de loin la plus large. Le graphique avoue que cette
moyenne ne veut pas dire grand-chose.

Avec `errorbar=None`, les six barres auraient l'air egalement solides.
C'est plus joli, et c'est moins honnete.

## 5.1 Le catalogue : quelle question, quel trace

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(17, 9))

# comparer -> des barres
sns.barplot(data=table, x="total", y="categorie", ax=axes[0, 0], color="#C1121F")
axes[0, 0].set_title("comparer -> barplot", loc="left", weight="bold")

# repartir -> un histogramme
sns.histplot(data=df, x="montant", bins=30, ax=axes[0, 1], color="#8B7B78")
axes[0, 1].set_title("repartir -> histplot", loc="left", weight="bold")

# comparer des formes -> une boite a moustaches
sns.boxplot(data=df, x="montant", y="categorie", ax=axes[0, 2], color="#8B7B78")
axes[0, 2].set_title("comparer des formes -> boxplot", loc="left", weight="bold")

# correler -> un nuage de points
avec_jour = df.assign(jour=df["date_operation"].dt.day)
sns.scatterplot(data=avec_jour, x="jour", y="montant", ax=axes[1, 0],
                alpha=0.6, color="#C1121F")
axes[1, 0].set_title("correler -> scatterplot", loc="left", weight="bold")

# evoluer -> une ligne
mensuel = df.groupby("mois")["montant"].sum().reset_index(name="total")
sns.lineplot(data=mensuel, x="mois", y="total", marker="o",
             ax=axes[1, 1], color="#C1121F")
axes[1, 1].set_ylim(0, mensuel["total"].max() * 1.1)
axes[1, 1].set_title("evoluer -> lineplot", loc="left", weight="bold")

# reperer -> une carte de chaleur
sns.heatmap(croise, ax=axes[1, 2], cmap="Reds", cbar=False)
axes[1, 2].set_title("reperer -> heatmap", loc="left", weight="bold")
axes[1, 2].tick_params(axis="y", rotation=0)
axes[1, 2].set_xlabel("")
axes[1, 2].set_ylabel("")

fig.tight_layout()
plt.show()

La planche a garder sous la main :

| La question | Le trace |
|---|---|
| comparer des quantites | `barplot` ou `ax.barh` |
| repartir une variable | `histplot` |
| comparer des formes | `boxplot` |
| correler deux variables | `scatterplot` |
| suivre dans le temps | `lineplot` |
| reperer une anomalie | `heatmap` |

Ce qui n'est **pas** dans la liste : le camembert. Au-dela de trois parts,
l'oeil compare mal des angles - et tres mal des angles vus en perspective.
Jamais de 3D.

La raison est mesurable, pas affaire de gout : on compare des **positions**
et des **longueurs** a quelques pourcents pres, des **angles** et des
**surfaces** beaucoup moins bien, et des **teintes** tres mal.

## 5.2 hue : une troisieme variable

In [ ]:
long = bilan.melt(id_vars="categorie",
                  value_vars=["reel", "budget_prevu"],
                  var_name="type", value_name="montant")
long = long.replace({"reel": "Depense", "budget_prevu": "Prevu"})

fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(data=long, x="montant", y="categorie", hue="type", ax=ax,
            palette={"Depense": "#C1121F", "Prevu": "#8B7B78"})
ax.set_title("Depense contre prevu, en euros : ou est le depassement ?",
             loc="left", weight="bold")
ax.set_xlabel("Sur six mois (EUR)")
ax.set_ylabel("")
ax.legend(title="")
fig.tight_layout()
plt.show()

`hue` a besoin de la forme **longue** : une colonne qui dit de quel groupe
il s'agit. C'est ce que fait `melt()`, l'inverse exact de `pivot_table`.

Maintenant regarde ce graphique et cherche le depassement des loisirs.

Il est **exact**, et il ne repond pas a la question. Le logement pese
36 000 euros et la sante 3 000 : les cinq petites categories sont ecrasees.

La solution n'est pas cosmetique. On ne change ni la taille, ni les
couleurs, ni l'axe : on change ce qu'on **encode**.

In [ ]:
tri = bilan.sort_values("ecart_pct")

fig, ax = plt.subplots(figsize=(10, 5))
couleurs = ["#C1121F" if e > 0 else "#2C8A1A" for e in tri["ecart_pct"]]
ax.barh(tri["categorie"], tri["ecart_pct"], color=couleurs)
ax.axvline(0, color="#444", linewidth=1.2)

pire = tri.iloc[-1]
ax.set_title(f"Les {pire['categorie'].lower()} depassent le budget "
             f"de {pire['ecart_pct']:.0f} %", loc="left", weight="bold")
ax.set_xlabel("Ecart au budget prevu (%)   -   negatif = tenu")
limite = tri["ecart_pct"].abs().max() * 1.35
ax.set_xlim(-limite, limite)
fig.tight_layout()
plt.show()

Memes donnees, meme calcul. Le pourcentage met toutes les categories sur la
meme echelle, et l'information apparait.

**Les deux graphiques sont exacts. Un seul repond a la question.**

## 5.3 Les facettes, et le piege de vocabulaire

In [ ]:
grille = sns.relplot(
    data=reel, x="mois", y="reel", col="categorie", col_wrap=3,
    kind="line", marker="o", height=2.8, aspect=1.3, color="#C1121F",
)
grille.set_titles("{col_name}")
grille.set_axis_labels("", "Total du mois (EUR)")
for un_axe in grille.axes.flat:
    un_axe.tick_params(axis="x", labelrotation=45)
plt.show()

Six graphiques, une ligne de code. Chaque categorie a son panneau, et tous
partagent la meme echelle : c'est ce qui rend la comparaison possible.

Le seul piege de vocabulaire de seaborn :

| Type | Exemples | Accepte `ax=` ? |
|---|---|---|
| axes-level | `barplot`, `histplot`, `lineplot`, `boxplot`, `heatmap` | oui |
| figure-level | `relplot`, `catplot`, `displot`, `lmplot` | **non** |

Les figure-level creent leur **propre** figure : c'est le prix des
facettes. D'ou l'erreur classique `sns.catplot(..., ax=axes[0, 0])` qui
leve un `TypeError`.

A retenir : si le nom est un `-plot` tout court, cela prend un `ax`. S'il a
un prefixe (`rel`, `cat`, `dis`, `lm`), non.

Pour un tableau de bord, on reste sur les axes-level.

## 5.4 La limite des sept couleurs

In [ ]:
print("moyen_paiement :", df["moyen_paiement"].nunique(), "valeurs")
print("libelle        :", df["libelle"].nunique(), "valeurs")

fig, (a, b) = plt.subplots(1, 2, figsize=(15, 5))

sns.scatterplot(data=avec_jour, x="jour", y="montant", hue="moyen_paiement",
                ax=a, alpha=0.7)
a.set_title("hue = moyen_paiement : lisible", loc="left", weight="bold")

sns.scatterplot(data=avec_jour, x="jour", y="montant", hue="libelle",
                ax=b, alpha=0.7, legend=False)
b.set_title("hue = libelle : illisible", loc="left", weight="bold")

fig.tight_layout()
plt.show()

La limite pratique est autour de **sept** couleurs. Au-dela, l'oeil ne fait
plus la difference entre deux teintes voisines, et la legende devient plus
grande que le graphique.

Que faire quand il y a trop de categories : regrouper les petites dans
« Autre », griser tout sauf les cinq principales, ou passer aux facettes.

---

# 6. Les cinq regles de la dataviz honnete

1. Le **titre porte le message**, pas la description.
2. L'axe des **barres part de zero**. Toujours.
3. **Trier** par valeur - sauf ce qui a un ordre naturel, comme les mois.
4. **Pas de camembert** au-dela de trois parts. Jamais en 3D.
5. **Une question, un graphique.** Si tu ne peux pas dire en une phrase ce
   que le graphique repond, il n'est pas pret.

## 6.1 La regle 1 en pratique

| Descriptif | Le message |
|---|---|
| Depenses par categorie | Le logement absorbe 61 % du budget a lui seul |
| Distribution des montants | 80 % des depenses sont sous 150 EUR |
| Reel contre prevu | Les loisirs depassent le budget de 30 % |
| Evolution mensuelle | 4 mois sur 6 depassent le budget prevu |

Dans la colonne de gauche, le lecteur doit faire le travail. Dans celle de
droite, il est fait.

**Le test** : si tu peux remplacer ton titre par le nom de la colonne de
l'axe sans rien perdre, ce n'est pas un titre.

## 6.2 Le non-resultat

Un graphique qui ne montre rien est un resultat. Le dire est plus utile que
de le cacher.

In [ ]:
r = avec_jour["montant"].corr(avec_jour["jour"])

fig, ax = plt.subplots(figsize=(9, 5))
sns.scatterplot(data=avec_jour, x="jour", y="montant", hue="categorie",
                alpha=0.7, ax=ax)
ax.set_title(f"Aucun lien entre le montant et le jour du mois (r = {r:+.2f})",
             loc="left", weight="bold")
ax.set_xlabel("Jour du mois")
ax.set_ylabel("Montant (EUR)")
ax.legend(title="", fontsize=8, ncol=2)
fig.tight_layout()
plt.show()

Trois reactions possibles quand un graphique ne montre rien :

| Reaction | Verdict |
|---|---|
| Jeter le graphique et n'en parler a personne | mauvaise |
| Tronquer les axes jusqu'a ce qu'une difference apparaisse | **pire** |
| Garder le graphique et **ecrire** le non-resultat dans le titre | bonne |

« Cette variable n'explique rien » est une information : elle dit a la
personne qui lira apres toi de ne pas perdre son temps a refaire l'analyse.

Un titre honnete peut dire qu'il n'y a **rien** a voir.

---

# 7. Le tableau de bord

Une figure, quatre panneaux, quatre questions.

Regle d'architecture : **chaque fonction de trace recoit son `ax`**. Une
fonction qui cree sa propre figure ne peut pas etre reutilisee dans une
grille. C'est la meme idee que « une fonction rend une valeur, elle
n'affiche pas » de la seance 4.

In [ ]:
ACCENT, NEUTRE, VERT = "#C1121F", "#8B7B78", "#2C8A1A"


def euros(x):
    """Formate un montant a la francaise : 36 251 EUR."""
    return f"{x:,.0f} EUR".replace(",", " ")


def panneau_categories(donnees, ax):
    """Panneau 1 : ou part l'argent ?"""
    t = (donnees.groupby("categorie")["montant"].sum()
         .sort_values(ascending=False).reset_index(name="total"))
    part = t["total"].iloc[0] / t["total"].sum() * 100
    ax.barh(t["categorie"], t["total"], color=[ACCENT] + [NEUTRE] * (len(t) - 1))
    ax.invert_yaxis()
    ax.set_title(f"Le {t['categorie'].iloc[0].lower()} absorbe {part:.0f} % "
                 "du budget a lui seul", loc="left", weight="bold")
    ax.set_xlabel("Total depense sur six mois (EUR)")
    ax.set_xlim(0, t["total"].max() * 1.22)
    for y, v in enumerate(t["total"]):
        ax.text(v + t["total"].max() * 0.02, y, euros(v), va="center", fontsize=9)
    return ax


def panneau_repartition(donnees, ax):
    """Panneau 2 : nombreuses et petites, ou rares et grosses ?"""
    mediane, moyenne = donnees["montant"].median(), donnees["montant"].mean()
    sous = (donnees["montant"] < 150).mean() * 100
    sns.histplot(data=donnees, x="montant", bins=30, ax=ax, color=NEUTRE)
    ax.axvline(mediane, color=ACCENT, linewidth=2, label=f"mediane {euros(mediane)}")
    ax.axvline(moyenne, color=ACCENT, linestyle="--", linewidth=2,
               label=f"moyenne {euros(moyenne)}")
    ax.set_title(f"{sous:.0f} % des depenses sont sous 150 EUR,\n"
                 "mais la moyenne est tiree par les loyers",
                 loc="left", weight="bold")
    ax.set_xlabel("Montant d'une depense (EUR)")
    ax.set_ylabel("Nombre de depenses")
    ax.legend()
    return ax


def panneau_ecart(table_bilan, ax):
    """Panneau 3 : ai-je tenu mon budget ? C'est le fruit de la jointure."""
    t = table_bilan.sort_values("ecart_pct")
    ax.barh(t["categorie"], t["ecart_pct"],
            color=[ACCENT if e > 0 else VERT for e in t["ecart_pct"]])
    ax.axvline(0, color="#444", linewidth=1.2)
    pire = t.iloc[-1]
    ax.set_title(f"Les {pire['categorie'].lower()} depassent le budget "
                 f"de {pire['ecart_pct']:.0f} %", loc="left", weight="bold")
    ax.set_xlabel("Ecart au budget prevu (%)   -   negatif = tenu")
    limite = t["ecart_pct"].abs().max() * 1.35
    ax.set_xlim(-limite, limite)
    return ax


def panneau_evolution(donnees, table_budget, ax):
    """Panneau 4 : est-ce que ca derape, et depuis quand ?"""
    reel_mois = donnees.groupby("mois")["montant"].sum().reset_index(name="total")
    prevu_mois = table_budget.groupby("mois")["budget_prevu"].sum().reset_index()
    suivi = reel_mois.merge(prevu_mois, on="mois", how="left")
    ax.plot(suivi["mois"], suivi["total"], marker="o", color=ACCENT,
            linewidth=2.5, label="Depense")
    ax.plot(suivi["mois"], suivi["budget_prevu"], marker="o", color=NEUTRE,
            linestyle="--", linewidth=2, label="Prevu")
    n = int((suivi["total"] > suivi["budget_prevu"]).sum())
    ax.set_title(f"{n} mois sur {len(suivi)} depassent le budget prevu",
                 loc="left", weight="bold")
    ax.set_ylabel("Total du mois (EUR)")
    # REGLE 2 : l'axe part de zero.
    ax.set_ylim(0, suivi[["total", "budget_prevu"]].to_numpy().max() * 1.15)
    ax.legend()
    return ax

In [ ]:
sns.set_theme(style="whitegrid", font_scale=1.05)

fig, axes = plt.subplots(2, 2, figsize=(16, 11))

panneau_categories(df, axes[0, 0])
panneau_repartition(df, axes[0, 1])
panneau_ecart(bilan, axes[1, 0])
panneau_evolution(df, budget, axes[1, 1])

total = f"{df['montant'].sum():,.0f}".replace(",", " ")
fig.suptitle(f"MonBudget - six mois, {len(df)} depenses, {total} EUR",
             fontsize=17, weight="bold")

# tight_layout AVANT savefig, et rect pour reserver la bande du suptitle.
fig.tight_layout(rect=(0, 0, 1, 0.97))
fig.savefig(FIGURES / "tableau_de_bord.png", dpi=150, bbox_inches="tight")
plt.show()

Deux details qui font la difference :

- `fig.tight_layout()` **avant** `savefig`, sinon les titres se
  chevauchent.
- le `rect=(0, 0, 1, 0.97)` reserve la bande du haut au `suptitle` ; sans
  lui, les titres de panneaux passent dessous.

Et la case vide, si tu passes a cinq panneaux : `axes[2, 1].axis("off")`.
Une case vide est plus honnete qu'un panneau ajoute pour remplir. La regle 5
interdit aussi le graphique sans question.

---

# 8. A toi

## 8.1 Les titres - l'exercice le plus important

Voici quatre titres descriptifs :

- Depenses par categorie
- Distribution des montants
- Reel contre prevu
- Evolution mensuelle

Pour chacun, regarde le panneau correspondant et reponds a la question :
**qu'est-ce que ce graphique m'apprend, en une phrase ?** Cette phrase est
le titre.

Compare ensuite avec les titres du code ci-dessus. Cet exercice ne demande
pas une ligne de Python, et c'est celui qui compte le plus.

## 8.2 Cinq questions

Ecris ta reponse dans la cellule qui suit chaque question.

**Q1.** Combien de couples (categorie, mois) sont en depassement, sur les
36 que compte la table ?

In [ ]:
# Ta reponse ici

**Q2.** Quelle categorie depasse son budget le plus **souvent**, en nombre
de mois ? Est-ce la meme que celle qui depasse le plus en euros ?

In [ ]:
# Ta reponse ici

**Q3.** Trace un `boxplot` des montants par mois. Que raconte-t-il que le
`lineplot` des totaux ne raconte pas ?

In [ ]:
# Ta reponse ici

**Q4.** Ajoute un cinquieme panneau au tableau de bord : la heatmap
categorie x mois. Faut-il passer en 2x3 ou en 3x2 ? Laquelle est la plus
lisible sur un videoprojecteur 16/9 ?

In [ ]:
# Ta reponse ici

**Q5.** Refais le `merge` avec `how="inner"` apres avoir retire une ligne du
budget. Combien de lignes perds-tu, et comment l'aurais-tu su sans les
compter ?

In [ ]:
# Ta reponse ici

---

## Ce qu'il faut retenir

1. **Une agregation seule ne se lit pas.** Il en faut deux, de natures
   differentes : un total et un compte, un niveau et une dispersion.
2. **Si le nombre de lignes a grossi apres un `merge`, c'est un bug**
   jusqu'a preuve du contraire.
3. **`how="left"` rend le trou visible ; `how="inner"` le cache.**
4. **Un tableau de bord n'est pas quatre graphiques**, c'est une figure qui
   en contient quatre - et c'est ce qui rend leurs echelles comparables.
5. **Un graphique est un encodage, pas une illustration.** Position et
   longueur se lisent bien ; angles et teintes, mal.
6. **Le choix de ce qu'on encode decide de ce que le lecteur pourra voir.**
7. **Le titre porte le message** - et l'ecrire est un test de completude de
   l'analyse.

## Les corriges

Les corriges du TD sont dans `seances/s08-dataviz/corrige/reprise/`.
Le corrige de reference du projet est dans `fil-rouge/v5-dashboard/`.

## Et apres ?

Quatre chemins pour continuer :

| Direction | Par ou commencer |
|---|---|
| Data et IA | scikit-learn, puis Kaggle Learn |
| Collecter | scraping et APIs publiques, pour fabriquer ses donnees |
| Exposer | FastAPI, pour passer du script au service |
| Automatiser | *Automate the Boring Stuff with Python* |

Le depot reste ouvert, et Classroom aussi.

## Bloque ?

Colle le message d'erreur **complet, en texte** - jamais une capture - dans
le flux du cours sur Google Classroom.